# Pipeline Tuteur Conversationnel Émotionnel — Étape 4
**Mémoire M1 — Zinedine Hamadi & Sara Hadidi — Sorbonne Université 2025/2026**

## Instructions
1. `Exécution` → `Modifier le type d'exécution` → **GPU T4**
2. Exécuter les cellules dans l'ordre (Shift+Entrée)
3. Le modèle EmoBERT doit être sauvegardé sur Google Drive dans `emobert_v1/`

## Structure
- **Étape 4.1** — EmotionClassifier (EmoBERT encapsulé)
- **Étape 4.2** — Stratégies pédagogiques (strategies.yaml)
- **Étape 4.3** — LLM Client (Mistral via Ollama ou OpenAI)
- **Étape 4.4** — Prompt Builder
- **Étape 4.5** — Pipeline complet (EmotionalTutor)
- **Étape 4.6** — Notebook de démonstration
- **Étape 4.7** — Évaluation du pipeline
- **Étape 4.8** — Export GitHub

## Cellule 0 — Installation & Google Drive

In [1]:
!pip install -q transformers torch pyyaml openai

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, datetime
import torch
import torch.nn.functional as F
import yaml
import requests
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('✓ Imports OK')

Mounted at /content/drive
GPU : Tesla T4
✓ Imports OK


In [2]:
import os

# Chercher le dossier emobert sur Drive
drive_path = '/content/drive/MyDrive'
for item in os.listdir(drive_path):
    if 'emobert' in item.lower():
        print(f'Trouvé : {item}')
        # Lister le contenu
        full_path = os.path.join(drive_path, item)
        if os.path.isdir(full_path):
            print(f'  Contenu : {os.listdir(full_path)}')

Trouvé : emobert_best_model
  Contenu : ['config.json', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'tokenizer.json']
Trouvé : emobert_6classes
  Contenu : ['checkpoint-25', 'checkpoint-125', 'best_model', 'checkpoint-50', 'checkpoint-100', 'checkpoint-75', 'confusion_matrix_6classes.png']
Trouvé : emobert_train.ipynb
Trouvé : emobert_v2
  Contenu : ['config.json', 'label_mapping.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'model.safetensors']


## Étape 4.1 — EmotionClassifier (EmoBERT encapsulé)

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/emobert_v2'
LABELS     = ['stress', 'frustration', 'engagement', 'confusion', 'satisfaction', 'neutre']
THRESHOLD  = 0.5
MAX_LENGTH = 128

# ── Classe EmotionClassifier ──────────────────────────────────────────────────
class EmotionClassifier:
    """
    Classifieur d'émotions éducatives basé sur RoBERTa fine-tuné.
    Interface stable : predict(text) -> dict
    """
    def __init__(self, model_path=MODEL_PATH):
        print(f'  Chargement EmoBERT depuis : {model_path}')
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.eval()
        print(f'  ✓ EmoBERT chargé sur {self.device}')

    def predict(self, text: str) -> dict:
        if not text or not text.strip():
            return {'label': 'neutre', 'confidence': 0.0,
                    'all_scores': {l: 0.0 for l in LABELS}}
        inputs = self.tokenizer(
            text, return_tensors='pt',
            truncation=True, max_length=MAX_LENGTH, padding=True
        ).to(self.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs      = F.softmax(logits, dim=-1).squeeze()
        confidence = probs.max().item()
        label_idx  = probs.argmax().item()
        label = LABELS[label_idx] if confidence >= THRESHOLD else 'neutre'
        return {
            'label':      label,
            'confidence': round(confidence, 3),
            'all_scores': {l: round(p.item(), 3) for l, p in zip(LABELS, probs)}
        }

    def predict_batch(self, texts: list) -> list:
        return [self.predict(t) for t in texts]

# ── Chargement ────────────────────────────────────────────────────────────────
classifier = EmotionClassifier()

# ── Test rapide ───────────────────────────────────────────────────────────────
result = classifier.predict('Je ne comprends rien à ce chapitre.')
print(f'\n  Test : {result}')

  Chargement EmoBERT depuis : /content/drive/MyDrive/emobert_v2


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

  ✓ EmoBERT chargé sur cuda

  Test : {'label': 'confusion', 'confidence': 0.594, 'all_scores': {'stress': 0.029, 'frustration': 0.119, 'engagement': 0.102, 'confusion': 0.594, 'satisfaction': 0.036, 'neutre': 0.118}}


In [4]:
# ── 6 Tests unitaires ─────────────────────────────────────────────────────────
test_cases = [
    ('stress',       "Je suis complètement dépassé, l'examen est demain et j'ai rien révisé."),
    ('frustration',  "J'ai encore faux ! Je comprends pas pourquoi ça marche pas."),
    ('engagement',   "C'est super intéressant ! Je veux vraiment comprendre ce concept."),
    ('confusion',    "Je vois pas du tout la différence entre les deux notions."),
    ('satisfaction', "Ah oui ! Maintenant c'est clair, j'ai enfin compris !"),
    ('neutre',       "Bonjour, pouvez-vous m'aider s'il vous plaît ?"),
]

print('=== Tests unitaires — EmotionClassifier ===')
for expected, text in test_cases:
    r = classifier.predict(text)
    assert isinstance(r, dict)
    assert r['label'] in LABELS
    assert 0.0 <= r['confidence'] <= 1.0
    assert len(r['all_scores']) == 6
    match = '✓ correct' if r['label'] == expected else f'→ prédit {r["label"]}'
    print(f'  [{expected:<15}] {match} ({r["confidence"]:.0%})')
    print(f'    "{text[:60]}"')

# Latence
start = time.time()
for _ in range(50): classifier.predict('Je suis perdu.')
lat = (time.time() - start) / 50 * 1000
print(f'\n  Latence moyenne : {lat:.1f} ms (cible < {"50" if torch.cuda.is_available() else "200"} ms)')
print('✓ Étape 4.1 OK')

=== Tests unitaires — EmotionClassifier ===
  [stress         ] → prédit engagement (58%)
    "Je suis complètement dépassé, l'examen est demain et j'ai ri"
  [frustration    ] → prédit engagement (75%)
    "J'ai encore faux ! Je comprends pas pourquoi ça marche pas."
  [engagement     ] ✓ correct (76%)
    "C'est super intéressant ! Je veux vraiment comprendre ce con"
  [confusion      ] → prédit engagement (73%)
    "Je vois pas du tout la différence entre les deux notions."
  [satisfaction   ] → prédit engagement (50%)
    "Ah oui ! Maintenant c'est clair, j'ai enfin compris !"
  [neutre         ] → prédit engagement (69%)
    "Bonjour, pouvez-vous m'aider s'il vous plaît ?"

  Latence moyenne : 31.1 ms (cible < 50 ms)
✓ Étape 4.1 OK


## Étape 4.2 — Stratégies pédagogiques (strategies.yaml)

In [5]:
# ── Créer strategies.yaml ─────────────────────────────────────────────────────
strategies_content = """stress:
  tone: "rassurant et bienveillant"
  instruction: >
    L'apprenant semble stressé. Adopte un ton très rassurant.
    Décompose ta réponse en petites étapes simples.
    Valide explicitement l'effort fourni avant d'expliquer.
    Évite les formulations complexes ou les listes longues.

frustration:
  tone: "empathique et concret"
  instruction: >
    L'apprenant semble frustré. Commence par reconnaître sa
    difficulté. Reformule l'explication avec un exemple concret.
    Évite le jargon technique. Propose une seule chose à la fois.

confusion:
  tone: "clair et structuré"
  instruction: >
    L'apprenant semble confus. Vérifie d'abord les prérequis.
    Utilise des analogies simples. Pose une question de
    vérification à la fin pour t'assurer de la compréhension.

engagement:
  tone: "dynamique et stimulant"
  instruction: >
    L'apprenant est engagé et motivé. Profite de ce moment pour
    approfondir. Propose un défi supplémentaire ou une question
    ouverte pour stimuler la réflexion.

satisfaction:
  tone: "encourageant et progressif"
  instruction: >
    L'apprenant est satisfait. Renforce positivement. Fais le
    lien avec la prochaine notion pour maintenir l'élan.

neutre:
  tone: "pédagogique et neutre"
  instruction: >
    Adopte un ton pédagogique standard. Réponds directement
    de manière claire et structurée.
"""

with open('strategies.yaml', 'w', encoding='utf-8') as f:
    f.write(strategies_content)

# ── Charger et tester ─────────────────────────────────────────────────────────
with open('strategies.yaml', 'r', encoding='utf-8') as f:
    STRATEGIES = yaml.safe_load(f)

def get_strategy(emotion: str) -> dict:
    return STRATEGIES.get(emotion, STRATEGIES['neutre'])

print('=== Stratégies chargées ===')
for emotion in LABELS:
    s = get_strategy(emotion)
    print(f'  {emotion:<15} → tone: "{s["tone"]}"')

print('\n✓ Étape 4.2 OK')

=== Stratégies chargées ===
  stress          → tone: "rassurant et bienveillant"
  frustration     → tone: "empathique et concret"
  engagement      → tone: "dynamique et stimulant"
  confusion       → tone: "clair et structuré"
  satisfaction    → tone: "encourageant et progressif"
  neutre          → tone: "pédagogique et neutre"

✓ Étape 4.2 OK


## Étape 4.3 — LLM Client (OpenAI GPT-4o-mini)

In [6]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

print("Chargement de Zephyr-7B sur GPU Colab... (~2 min)")

MODEL_LLM = "HuggingFaceH4/zephyr-7b-beta"

pipe = pipeline(
    "text-generation",
    model=MODEL_LLM,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("✓ Modèle LLM chargé")

def generate(system_prompt: str, user_message: str, history: list = []) -> str:
    messages = [{"role": "system", "content": system_prompt}]
    messages += history
    messages.append({"role": "user", "content": user_message})

    output = pipe(
        messages,
        max_new_tokens=300,
    )
    return output[0]["generated_text"][-1]["content"].strip()

# Test
print("\nTest LLM...")
test_response = generate(
    system_prompt="Tu es un tuteur pédagogique bienveillant. Réponds en français.",
    user_message="Bonjour, peux-tu m'expliquer ce qu'est un arbre binaire ?"
)
print(test_response)
print("\n✓ Étape 4.3 OK")

Chargement de Zephyr-7B sur GPU Colab... (~2 min)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Modèle LLM chargé

Test LLM...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Bien sûr ! Un arbre binaire, également appelé arbre de recherche, est un type particulier d'arbre qui permet d'effectuer des opérations de recherche, d'insertion ou de suppression d'éléments en temps constant moyen, autrement dit, en temps qui ne varie pas en fonction de la taille de l'arbre. 

Dans un arbre binaire, chaque nœud (sommet) possède au maximum deux enfants, appelés les enfants gauche et droit, et est donc doté d'un seul parent. Les feuilles (sommets sans enfant) d'un arbre binaire sont les éléments que l'on souhaite rechercher, insérer ou supprimer.

L'ordre dans lequel les éléments sont insérés ou supprimés dans l'arbre binaire détermine l'emplacement des feuilles dans l'arbre. Les éléments sont souvent stockés selon un certain ordre, par exemple une clé, afin de les retrouver facilement.

J'espère que cela vous aide à mieux comprendre l'arbre binaire ! Let me know if you have any further questions.

✓ Étape 4.3 OK


## Étape 4.4 — Prompt Builder

In [7]:
# ── Prompt système de base ────────────────────────────────────────────────────
SYSTEM_BASE = """Tu es un tuteur pédagogique intelligent spécialisé en informatique
et sciences du langage. Ton rôle est d'aider les apprenants à comprendre des concepts
complexes de manière adaptée à leur état émotionnel.
Tu t'exprimes toujours en français, de manière claire et bienveillante.

{emotional_instruction}

Contexte émotionnel détecté : {emotion} (confiance : {confidence:.0%})
"""

def build_prompt(emotion: str, confidence: float) -> str:
    """
    Construit le system prompt en injectant la stratégie pédagogique
    correspondant à l'émotion détectée.
    """
    instruction = STRATEGIES.get(emotion, STRATEGIES['neutre'])['instruction']
    return SYSTEM_BASE.format(
        emotional_instruction=instruction,
        emotion=emotion,
        confidence=confidence
    )

# ── Tests : 6 cas (un par classe) ─────────────────────────────────────────────
print('=== Tests Prompt Builder ===')
test_prompts = [
    ('confusion',    0.81),
    ('engagement',   0.92),
    ('stress',       0.75),
    ('frustration',  0.68),
    ('satisfaction', 0.88),
    ('neutre',       0.38),
]
for emotion, conf in test_prompts:
    prompt = build_prompt(emotion, conf)
    assert emotion in prompt or 'neutre' in prompt
    print(f'  ✓ [{emotion:<15}] prompt généré ({len(prompt)} chars)')

# Afficher un exemple
print('\n--- Exemple prompt (confusion, 81%) ---')
print(build_prompt('confusion', 0.81))
print('\n✓ Étape 4.4 OK')

=== Tests Prompt Builder ===
  ✓ [confusion      ] prompt généré (507 chars)
  ✓ [engagement     ] prompt généré (496 chars)
  ✓ [stress         ] prompt généré (554 chars)
  ✓ [frustration    ] prompt généré (520 chars)
  ✓ [satisfaction   ] prompt généré (453 chars)
  ✓ [neutre         ] prompt généré (425 chars)

--- Exemple prompt (confusion, 81%) ---
Tu es un tuteur pédagogique intelligent spécialisé en informatique
et sciences du langage. Ton rôle est d'aider les apprenants à comprendre des concepts
complexes de manière adaptée à leur état émotionnel.
Tu t'exprimes toujours en français, de manière claire et bienveillante.

L'apprenant semble confus. Vérifie d'abord les prérequis. Utilise des analogies simples. Pose une question de vérification à la fin pour t'assurer de la compréhension.


Contexte émotionnel détecté : confusion (confiance : 81%)


✓ Étape 4.4 OK


## Étape 4.5 — Pipeline complet (EmotionalTutor)

In [8]:
os.makedirs('logs', exist_ok=True)

# ── Gestion de l'historique ───────────────────────────────────────────────────
class DialogueHistory:
    def __init__(self, max_turns: int = 5):
        self.history   = []
        self.max_turns = max_turns

    def add(self, role: str, content: str):
        self.history.append({'role': role, 'content': content})
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]

    def get(self):
        return self.history.copy()

    def reset(self):
        self.history = []

# ── Classe principale EmotionalTutor ─────────────────────────────────────────
class EmotionalTutor:
    """
    Pipeline complet : EmoBERT → PromptBuilder → LLM → Logger
    """
    def __init__(self):
        self.classifier = classifier   # EmotionClassifier chargé à l'étape 4.1
        self.history    = DialogueHistory(max_turns=5)
        self.logs       = []
        print('✓ EmotionalTutor initialisé')

    def respond(self, user_message: str) -> dict:
        # 1. Détection émotionnelle
        emotion_result = self.classifier.predict(user_message)
        emotion        = emotion_result['label']
        confidence     = emotion_result['confidence']

        # 2. Construction du prompt adaptatif
        system_prompt = build_prompt(emotion, confidence)

        # 3. Génération de la réponse LLM
        response = generate(system_prompt, user_message, self.history.get())

        # 4. Mise à jour de l'historique
        self.history.add('user',      user_message)
        self.history.add('assistant', response)

        # 5. Logging
        log_entry = {
            'timestamp':    datetime.datetime.now().isoformat(),
            'user_message': user_message,
            'emotion':      emotion,
            'confidence':   confidence,
            'all_scores':   emotion_result['all_scores'],
            'llm_response': response
        }
        self.logs.append(log_entry)
        with open('logs/dialogues.jsonl', 'a', encoding='utf-8') as f:
            f.write(json.dumps(log_entry, ensure_ascii=False) + '\n')

        return {'response': response, 'emotion': emotion, 'confidence': confidence}

    def reset(self):
        self.history.reset()
        print('  Historique réinitialisé')

# ── Initialisation ────────────────────────────────────────────────────────────
tutor = EmotionalTutor()
print('\n✓ Étape 4.5 OK — Pipeline prêt')

✓ EmotionalTutor initialisé

✓ Étape 4.5 OK — Pipeline prêt


## Étape 4.6 — Démonstration sur 5 dialogues

In [9]:
tutor.reset()

def demo_turn(message: str):
    result = tutor.respond(message)
    print(f'Apprenant : {message}')
    print(f'Émotion   : {result["emotion"]} ({result["confidence"]:.0%})')
    print(f'Tuteur    : {result["response"]}')
    print('-' * 70)

print('=== DÉMONSTRATION — Scénario : arbres binaires ===')
print()

# Scénario couvrant les 6 émotions
demo_turn("Je ne comprends pas du tout ce qu'est un arbre binaire.")      # confusion
demo_turn("J'ai essayé 5 fois et mon code plante encore.")                # frustration
demo_turn("Super ! J'ai réussi l'insertion !")                            # satisfaction
demo_turn("C'est fascinant, je veux comprendre les arbres AVL maintenant.") # engagement
demo_turn("Je suis trop stressé par le partiel de demain.")               # stress

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Historique réinitialisé
=== DÉMONSTRATION — Scénario : arbres binaires ===



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant : Je ne comprends pas du tout ce qu'est un arbre binaire.
Émotion   : neutre (34%)
Tuteur    : Un arbre binaire, dans le contexte de l'informatique, est une structure de données qui permet de stocker et d'organiser des données de manière hiérarchique. Chaque nœud de cet arbre a au maximum deux enfants, appelés fils gauche et fils droit. Les valeurs ou les données sont stockées à chaque nœud, et elles sont réparties de manière récursive à travers l'arbre suivant la position des fils gauche et droit.

Imaginons qu'un arbre binaire soit représenté par un dessin : les nœuds seraient des cercles ou des carrés, et les valeurs ou les données seraient placées à l'intérieur de ces nœuds. Les fils gauche et droit seraient des lignes qui partent du nœud parent et se connectent au nœud fils gauche ou droit. Les arbres binaires sont utilisés dans plusieurs applications en informatique, comme les structures de données de tri, les recherches binaires ou les fichiers de compression.

L'idée 

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant : J'ai essayé 5 fois et mon code plante encore.
Émotion   : engagement (77%)
Tuteur    : Je suis désolé pour votre expérience difficile. Pour comprendre pourquoi votre code plante, il serait possible de commencer par vérifier les conditions préalables pour cet algorithme. Assurez-vous que les données sont correctement saisies et que les variables sont initialisées.

Pour identifier le problème, vous pouvez essayer de débugger votre code. Vous pouvez mettre des échos ou des affichages dans votre code pour vérifier les valeurs des variables au fur et à mesure de l'exécution. Cela vous permettra de suivre l'écoulement de votre code et de déterminer à quelle étape il se bloque.

Vous pouvez également essayer de simplifier votre code pour le rendre moins complexe. Le principe est de vérifier que chaque fonctionnalité soit fonctionnelle avant de les combiner. Cela vous aidera à identifier rapidement les problèmes.

Enfin, vous pouvez chercher des ressources en ligne pour obtenir de

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant : Super ! J'ai réussi l'insertion !
Émotion   : neutre (47%)
Tuteur    : Je suis heureux d'entendre que vous avez réussi l'insertion. Cela doit vous faire plaisir et vous motiver pour continuer à apprendre l'informatique. 

Pour continuer à progresser, vous pouvez essayer d'implémenter d'autres opérations sur les arbres binaires, tels que la recherche ou la suppression. Ces opérations sont également importantes pour comprendre les arbres binaires et leur utilisation dans les applications informatiques.

J'espère que vous continuez à trouver l'apprentissage de l'informatique stimulant et que vous n'hésitez pas à me contacter en cas de question.

Bonne chance avec vos futurs projets informatiques !
----------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant : C'est fascinant, je veux comprendre les arbres AVL maintenant.
Émotion   : engagement (76%)
Tuteur    : Les arbres AVL sont un type d'arbre binaire auto-équilibré, qui garantit une balance faible entre les nœuds de gauche et de droite. Les arbres AVL sont utilisés pour maintenir une structure de données efficace et rapide, même lors de multiples opérations d'insertion, de suppression et de recherche.

Le principe des arbres AVL est de maintenir une balance faible entre les hauteurs de l'arbre en modifiant la structure de l'arbre lorsque les hauteurs diffèrent de plus de 1. Lorsqu'un nœud est inséré, supprimé ou recherché, l'arbre AVL est rééquilibré en effectuant des rotations sur les nœuds pour maintenir la balance.

Voici une présentation simplifiée de l'implémentation d'un arbre AVL :

- Un nœud AVL est caractérisé par deux valeurs, la hauteur et la balance. La hauteur est la longueur du plus long chemin entre le nœud et ses feuilles enfants, et la balance est la différe

In [10]:
# ── Analyse des logs ──────────────────────────────────────────────────────────
import pandas as pd

logs = [json.loads(l) for l in open('logs/dialogues.jsonl', encoding='utf-8')]
df_logs = pd.DataFrame(logs)[['user_message', 'emotion', 'confidence', 'timestamp']]
df_logs['confidence'] = df_logs['confidence'].apply(lambda x: f'{x:.0%}')
print('=== Analyse des logs ===')
print(df_logs.to_string(index=False))
print(f'\n  Total tours : {len(logs)}')
print(f'  Distribution émotions :')
print(pd.Series([l["emotion"] for l in logs]).value_counts().to_string())

=== Analyse des logs ===
                                                  user_message    emotion confidence                  timestamp
       Je ne comprends pas du tout ce qu'est un arbre binaire.     neutre        34% 2026-06-07T05:56:02.470677
                 J'ai essayé 5 fois et mon code plante encore. engagement        77% 2026-06-07T05:58:13.723929
                             Super ! J'ai réussi l'insertion !     neutre        47% 2026-06-07T05:59:37.400269
C'est fascinant, je veux comprendre les arbres AVL maintenant. engagement        76% 2026-06-07T06:01:53.961895
                Je suis trop stressé par le partiel de demain.     stress        52% 2026-06-07T06:04:10.641673

  Total tours : 5
  Distribution émotions :
neutre        2
engagement    2
stress        1


## Étape 4.7 — Évaluation du pipeline (LLM-as-judge)

In [11]:
tutor.reset()

# ── Dialogues de test pour l'évaluation ──────────────────────────────────────
eval_dialogues = [
    "Je ne comprends pas du tout la récursivité.",
    "J'ai encore faux sur cet exercice, c'est décourageant.",
    "J'adore ce concept, je veux en savoir plus !",
    "Je suis complètement perdu dans ce cours.",
    "Super, j'ai enfin compris les pointeurs !",
    "Je suis stressé, l'examen c'est demain.",
    "Pouvez-vous m'expliquer les listes chaînées ?",
    "Je n'arrive pas à déboguer ce code depuis 2 heures.",
    "C'est vraiment intéressant cette notion de complexité !",
    "Je crois que j'ai compris, merci beaucoup !",
]

# ── Générer réponses AVEC injection émotionnelle ──────────────────────────────
print('Génération des réponses AVEC injection émotionnelle...')
results_with = []
for msg in eval_dialogues:
    r = tutor.respond(msg)
    results_with.append({'message': msg, 'emotion': r['emotion'],
                         'confidence': r['confidence'], 'response': r['response']})
    print(f'  [{r["emotion"]:<15}] {msg[:50]}...')

# ── Générer réponses SANS injection (baseline) ───────────────────────────────
print('\nGénération des réponses SANS injection (baseline)...')
BASELINE_PROMPT = """Tu es un tuteur pédagogique. Réponds en français de manière claire et structurée."""
results_without = []
for msg in eval_dialogues:
    resp = generate(BASELINE_PROMPT, msg)
    results_without.append({'message': msg, 'response': resp})
    print(f'  {msg[:50]}...')

print('\n✓ Réponses générées')

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Historique réinitialisé
Génération des réponses AVEC injection émotionnelle...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [neutre         ] Je ne comprends pas du tout la récursivité....


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Pouvez-vous m'expliquer les listes chaînées ?...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Je n'arrive pas à déboguer ce code depuis 2 heures...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  C'est vraiment intéressant cette notion de complex...
  Je crois que j'ai compris, merci beaucoup !...

✓ Réponses générées


In [12]:
# ── LLM-as-judge : évaluation automatique ────────────────────────────────────
JUDGE_PROMPT = """Tu es un expert en pédagogie. Évalue cette réponse de tuteur selon 3 critères.
Réponds UNIQUEMENT avec un JSON valide, sans texte avant ou après.
Format exact : {{"adequation_emotionnelle": X, "pertinence_pedagogique": X, "coherence": X}}
Chaque score est un entier de 1 à 5.

Message de l'apprenant : {message}
Émotion détectée : {emotion}
Réponse du tuteur : {response}

Critères :
- adequation_emotionnelle (1-5) : le ton correspond-il à l'émotion détectée ?
- pertinence_pedagogique (1-5) : la réponse aide-t-elle l'apprenant à progresser ?
- coherence (1-5) : la réponse est-elle claire et bien structurée ?"""

def evaluate_response(message, emotion, response):
    prompt = JUDGE_PROMPT.format(message=message, emotion=emotion, response=response)
    raw = generate('Tu es un évaluateur expert.', prompt)
    try:
        return json.loads(raw.strip())
    except:
        return {'adequation_emotionnelle': 0, 'pertinence_pedagogique': 0, 'coherence': 0}

print('Évaluation en cours (LLM-as-judge)...')
scores_with, scores_without = [], []

for rw, rwo in zip(results_with, results_without):
    sw  = evaluate_response(rw['message'],  rw['emotion'], rw['response'])
    swo = evaluate_response(rwo['message'], 'neutre',      rwo['response'])
    scores_with.append(sw)
    scores_without.append(swo)
    print(f'  Avec: {sw}  |  Sans: {swo}')

print('\n✓ Évaluation terminée')

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Évaluation en cours (LLM-as-judge)...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Avec: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}  |  Sans: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Avec: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}  |  Sans: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Avec: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}  |  Sans: {'adequation_emotionnelle': 0, 'pertinence_pedagogique': 0, 'coherence': 0}


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Avec: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}  |  Sans: {'adequation_emotionnelle': 5, 'pertinence_pedagogique': 5, 'coherence': 5}

✓ Évaluation terminée


In [13]:
# ── Tableau de résultats ──────────────────────────────────────────────────────
import numpy as np

def avg(scores, key):
    vals = [s[key] for s in scores if s[key] > 0]
    return round(np.mean(vals), 2) if vals else 0.0

print('\n=== TABLEAU DE RÉSULTATS ===')
print(f'{"Condition":<35} {"Adéquat. émot.":>15} {"Pertinence péd.":>16} {"Cohérence":>10}')
print('-' * 80)
print(f'{"Sans injection (baseline)":<35} '
      f'{avg(scores_without, "adequation_emotionnelle"):>15} '
      f'{avg(scores_without, "pertinence_pedagogique"):>16} '
      f'{avg(scores_without, "coherence"):>10}')
print(f'{"Avec injection (EmoBERT F1=0.20)":<35} '
      f'{avg(scores_with, "adequation_emotionnelle"):>15} '
      f'{avg(scores_with, "pertinence_pedagogique"):>16} '
      f'{avg(scores_with, "coherence"):>10}')

# Sauvegarder le rapport
rapport = {
    'sans_injection': {
        'adequation_emotionnelle': avg(scores_without, 'adequation_emotionnelle'),
        'pertinence_pedagogique':  avg(scores_without, 'pertinence_pedagogique'),
        'coherence':               avg(scores_without, 'coherence')
    },
    'avec_injection': {
        'adequation_emotionnelle': avg(scores_with, 'adequation_emotionnelle'),
        'pertinence_pedagogique':  avg(scores_with, 'pertinence_pedagogique'),
        'coherence':               avg(scores_with, 'coherence')
    }
}
with open('evaluation_pipeline.json', 'w', encoding='utf-8') as f:
    json.dump(rapport, f, indent=2, ensure_ascii=False)
print('\n✓ evaluation_pipeline.json sauvegardé')
print('\n✓ Étape 4.7 OK')


=== TABLEAU DE RÉSULTATS ===
Condition                            Adéquat. émot.  Pertinence péd.  Cohérence
--------------------------------------------------------------------------------
Sans injection (baseline)                       5.0              5.0        5.0
Avec injection (EmoBERT F1=0.20)                5.0              5.0        5.0

✓ evaluation_pipeline.json sauvegardé

✓ Étape 4.7 OK


## Étape 4.8 — Export GitHub (sauvegarde sur Drive)

In [14]:
# ── Créer tous les fichiers pour GitHub ───────────────────────────────────────
os.makedirs('pipeline', exist_ok=True)

# emotion_classifier.py
emotion_classifier_code = '''
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

LABELS    = ["stress", "frustration", "engagement", "confusion", "satisfaction", "neutre"]
THRESHOLD = 0.5

class EmotionClassifier:
    def __init__(self, model_path="emobert_v1/"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def predict(self, text: str) -> dict:
        if not text or not text.strip():
            return {"label": "neutre", "confidence": 0.0, "all_scores": {l: 0.0 for l in LABELS}}
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True).to(self.device)
        with torch.no_grad():
            probs = F.softmax(self.model(**inputs).logits, dim=-1).squeeze()
        confidence = probs.max().item()
        label = LABELS[probs.argmax().item()] if confidence >= THRESHOLD else "neutre"
        return {"label": label, "confidence": round(confidence, 3),
                "all_scores": {l: round(p.item(), 3) for l, p in zip(LABELS, probs)}}
'''

# prompt_builder.py
prompt_builder_code = '''
import yaml
with open("strategies.yaml", "r", encoding="utf-8") as f:
    STRATEGIES = yaml.safe_load(f)

SYSTEM_BASE = """Tu es un tuteur pédagogique intelligent spécialisé en informatique et sciences du langage.
Tu t\'exprimes toujours en français, de manière claire et bienveillante.
{emotional_instruction}
Contexte émotionnel détecté : {emotion} (confiance : {confidence:.0%})
"""

def build_prompt(emotion: str, confidence: float) -> str:
    instruction = STRATEGIES.get(emotion, STRATEGIES["neutre"])["instruction"]
    return SYSTEM_BASE.format(emotional_instruction=instruction, emotion=emotion, confidence=confidence)
'''

# llm_client.py
llm_client_code = '''
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def generate(system_prompt: str, user_message: str, history: list = []) -> str:
    messages = [{"role": "system", "content": system_prompt}]
    messages += history
    messages.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, max_tokens=300, temperature=0.7)
    return response.choices[0].message.content
'''

# pipeline.py
pipeline_code = '''
from emotion_classifier import EmotionClassifier
from prompt_builder import build_prompt
from llm_client import generate
import json, datetime, os

class DialogueHistory:
    def __init__(self, max_turns=5):
        self.history = []; self.max_turns = max_turns
    def add(self, role, content):
        self.history.append({"role": role, "content": content})
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]
    def get(self): return self.history.copy()
    def reset(self): self.history = []

class EmotionalTutor:
    def __init__(self):
        self.classifier = EmotionClassifier()
        self.history    = DialogueHistory()
        self.logs       = []

    def respond(self, user_message: str) -> dict:
        er      = self.classifier.predict(user_message)
        prompt  = build_prompt(er["label"], er["confidence"])
        resp    = generate(prompt, user_message, self.history.get())
        self.history.add("user", user_message)
        self.history.add("assistant", resp)
        entry = {"timestamp": datetime.datetime.now().isoformat(),
                 "user_message": user_message, "emotion": er["label"],
                 "confidence": er["confidence"], "all_scores": er["all_scores"],
                 "llm_response": resp}
        os.makedirs("logs", exist_ok=True)
        with open("logs/dialogues.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\\n")
        self.logs.append(entry)
        return {"response": resp, "emotion": er["label"], "confidence": er["confidence"]}

if __name__ == "__main__":
    tutor = EmotionalTutor()
    print("=== Tuteur Conversationnel Émotionnel ===")
    while True:
        msg = input("Apprenant : ").strip()
        if msg.lower() == "quit": break
        r = tutor.respond(msg)
        print(f"[{r[\'emotion\']} {r[\'confidence\']:.0%}] Tuteur : {r[\'response\']}\\n")
'''

# requirements.txt
requirements = """transformers>=4.40.0
torch>=2.0.0
pyyaml>=6.0
openai>=1.0.0
pandas>=2.0.0
numpy>=1.24.0
"""

# .gitignore
gitignore = """emobert_v1/
logs/
__pycache__/
*.pyc
.env
*.joblib
hf_cache/
venv/
"""

# Écrire tous les fichiers
files = {
    'pipeline/emotion_classifier.py': emotion_classifier_code,
    'pipeline/prompt_builder.py':     prompt_builder_code,
    'pipeline/llm_client.py':         llm_client_code,
    'pipeline/pipeline.py':           pipeline_code,
    'pipeline/strategies.yaml':       open('strategies.yaml').read(),
    'pipeline/requirements.txt':      requirements,
    'pipeline/.gitignore':            gitignore,
}

for path, content in files.items():
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content.strip())
    print(f'  ✓ {path}')

# Sauvegarder sur Drive
import shutil
drive_dest = '/content/drive/MyDrive/memoire_M1/pipeline_v1'
if os.path.exists(drive_dest):
    shutil.rmtree(drive_dest)
shutil.copytree('pipeline', drive_dest)
print(f'\n✓ Tous les fichiers sauvegardés sur Drive : {drive_dest}')
print('\n✓ Étape 4.8 OK — Prêt pour GitHub')

  ✓ pipeline/emotion_classifier.py
  ✓ pipeline/prompt_builder.py
  ✓ pipeline/llm_client.py
  ✓ pipeline/pipeline.py
  ✓ pipeline/strategies.yaml
  ✓ pipeline/requirements.txt
  ✓ pipeline/.gitignore

✓ Tous les fichiers sauvegardés sur Drive : /content/drive/MyDrive/memoire_M1/pipeline_v1

✓ Étape 4.8 OK — Prêt pour GitHub
